# 0. Introduction: how we got to Kubernetes

← [Start here](README.md) · next: [Orientation](01-orientation.ipynb)

Every later page assumes you already know *why* CIRRUS is built from containers,
Kubernetes and git. This page is that why. It starts from a problem you may well
have, walks through the four ways people have tried to solve it, and arrives at
the platform one step at a time — each option fixing what the last one could
not.

There is nothing to run here and no sign-in needed. If you already know why
Kubernetes exists, skip to [page 1](01-orientation.ipynb).

---

## The problem: something that has to stay up

As part of your research, you want to host something that has to *stay up*.

NCAR's HPC systems are built around a batch scheduler, and they are very good at
what a scheduler does: **you** decide when a job runs, it starts, it uses its
resources, and it ends. Jobs are not designed to run forever.

A different kind of research output works the other way round. **Someone else**
decides when it is needed, and it has to already be running when they ask:

| workload | who decides it runs |
| --- | --- |
| an interactive dashboard | a person clicks |
| an API or data endpoint | another machine or program asks |
| an ingest pipeline | new data lands at 3 a.m. |
| a database or catalog | its state has to persist between requests |
| CI workflows | someone pushes a code change |

**GDEX**, the Geoscience Data Exchange, is a good example. It has to be up all the
time so people can reach the data their research depends on. Over the years GDEX
and the services before it have run on every option below, and have run into
each one's strengths and weaknesses.

So what are your options?

---

## Option 1: buy your own server

**A server is a computer that serves other computers**, called *clients*, by
answering their requests. It can be specialised rack hardware, an old desktop, a
mini PC, or a Raspberry Pi — if it answers requests from other systems, it is a
server.

Servers are the foundation of everything on this page. They never go away; every
later option adds layers of software *on top of* servers to hide some of the
difficulty of running one. Even "the cloud" is a collection of servers, with
software and staff behind the scenes keeping it all running.

| what you get | what it costs you |
| --- | --- |
| total control | operational maintenance, forever |
| no queue, no quota, nobody else's policy | you set up the reverse proxy, the certificates, the DNS name |
| no new concepts to learn | it sits mostly idle |
| it will run anything you want | there is no second copy of anything |
| | OS updates can break your workload |

The right-hand column is the part that matters. If your focus is research, you
should not also have to be a system administrator. Updates matter because an
unpatched machine is an open door, but each update risks a package conflict that
breaks your workload for a few days. So, in practice, the updates stop.

> **How it actually goes.** A grant buys a server in 2016. In 2017 a paper cites
> its URL. In 2019 there is staff turnover, nobody really knows how to manage it,
> and the only time it is touched is when it is down. In 2026 it is still
> running, still cited, and still on its original operating system.

**Where that leaves the workload:** on one machine that nobody dares touch,
because if it breaks it will be very hard to get back.

---

## Option 2: a virtual machine (VM)

**Virtualisation is software that turns one physical computer into several
virtual ones.** Each virtual machine has its own operating system and runs as if
it were a separate server, but they all share the same hardware underneath.

The software that does this is called a **hypervisor**. Install it on several
servers and they become a **cluster**: one pool of hardware that hosts many VMs.
If a server dies, the hypervisor starts that server's VMs again on the machines
that are still up, as long as there is room:

```
  before                              after server 2 fails

  server 1  [ VM A ][ VM B ]          server 1  [ VM A ][ VM B ][ VM C ]
  server 2  [ VM C ][ VM D ]   ──▶    server 2   ✗  down
  server 3  [ VM E ]                  server 3  [ VM E ][ VM D ]
```

VMs are also just files, so you can save the state of a whole machine at one
moment — a **snapshot** — and go back to it if an upgrade goes wrong.

| what it fixes | what it still costs you |
| --- | --- |
| one machine becomes many, and the hardware is better used | you still own a full operating system, and its updates |
| a hardware failure stops being fatal | failover is disruptive: the VM comes back as if it had been rebooted |
| snapshots can undo a bad change | adding CPU or memory often means powering the VM off |
| a real system administrator can look after the hardware | the workload is still tied to the OS under it, so OS upgrades can still break it |

**Where that leaves the workload:** it survives a dead server automatically, but
with a short outage. Someone still has to maintain the VM, and updates can still
cause version mismatches that break it.

Notice what is now in the way. You do not care about the operating system; you
care about your workload. The OS was only ever there because the workload needed
something to run on. If the workload did not depend on the OS, most of the
remaining problems would go with it.

---

## Option 3: put it in a container

The name comes from shipping, and the analogy is exact.

Before standard shipping containers, every cargo was loaded differently. The
standard container fixed that: one size, one shape, the same fittings, so every
crane, ship, train and truck in the world handles it the same way — whether it
holds pens or Porsches.

**A software container applies the same idea to software.** It packages your
code together with everything it needs to run — the language, the libraries,
the settings — behind one standard interface. The computer running it does not
need to know what is inside, and what is inside does not depend on the computer.

Three words you will meet constantly:

* A **Dockerfile** (also called a *Containerfile*) is a short text recipe: start
  from this base, install these packages, copy in this code, run this command.
* An **image** is what you build from that recipe — a packaged, unchangeable
  snapshot of your workload, ready to run.
* A **container registry** is an online library of images. You push an image to
  it once; any machine can then pull it by name.

```
  Dockerfile  ──build──▶  image  ──push──▶  registry  ──pull──▶  runs anywhere
  (the recipe)            (the package)     (the library)         laptop, server, cloud
```

Because the image carries its own requirements, it runs the same on your
laptop, a colleague's laptop, a server or a public cloud. If it worked for you,
it works for everyone.

| what it fixes | what it still costs you |
| --- | --- |
| the workload carries its own requirements | it still runs on one machine |
| server updates no longer touch the workload | a container that dies stays dead until somebody notices |
| the Dockerfile is a record of exactly how it was built | running more copies means setting up a load balancer yourself |
| the same image runs on your laptop and on the server | someone still has to maintain the VM it runs on |

**Where that leaves the workload:** easy to reproduce, but still on a single
server. If that server goes down, so does everything on it, and someone has to
start it all again by hand.

The workload is in good shape now. What needs to improve is the thing *hosting*
it.

---

## Option 4: Kubernetes

What is missing is something that watches all the running containers, across
many servers, makes sure everything that should be running *is* running, and
replaces anything that dies. That job is called **orchestration**.

**Kubernetes** is a container orchestrator. Google released it in 2015 and gave
it to the Cloud Native Computing Foundation, which looks after it today. The name
is Greek for *helmsman* — hence the ship's wheel in its logo — and it is often
shortened to **K8s** (K, eight letters, s).

You stop telling individual machines what to do. Instead you **describe what you
want** — "run three copies of this image, give each 2 CPUs and 4 GB of memory,
and make it reachable at this address" — and Kubernetes decides which servers
they run on and keeps it that way. It spreads requests across the copies for
you.

When a server fails, the copies on the other servers keep serving while
Kubernetes starts replacements, so there is no outage to wait out:

```
  before                              after server 2 fails

  server 1  [ app ][ db  ]            server 1  [ app ][ db  ][ api ]
  server 2  [ app ][ api ]     ──▶    server 2   ✗  down
  server 3  [ app ][ web ]            server 3  [ app ][ web ][ app ]

  three copies of "app" were asked for, so a third one is started on server 3
  while the other two carry on serving. Nobody was paged.
```

| what it fixes | what it still costs you |
| --- | --- |
| the workload is no longer tied to one machine | everything is described in YAML (below) |
| maintenance becomes mostly invisible: a server can be drained and patched while workloads move off it | there is a real learning curve |
| capacity is pooled across all the servers | somebody has to run the cluster itself |
| you declare what you want and the cluster keeps it true | |
| shared services can automate certificates, DNS names, and similar chores | |

The descriptions Kubernetes reads are written in **YAML**, a plain-text format
for configuration. It is mostly indentation and `key: value` pairs:

```yaml
replicas: 3            # how many copies
image: hub.k8s.ucar.edu/my-project/my-dashboard:v1.2
resources:
  cpu: "2"
  memory: 4Gi
```

Groups of these files that make up one application can be bundled into a
**Helm chart**, so it can be installed, configured and removed as one thing.

**Where that leaves the workload:** reproducible, because it is built from an
image and described in YAML. It stays up when servers go down. And the system
maintenance is somebody else's job.

That last cost — *somebody has to run the cluster* — is what the **CIRRUS team**
is for. The team runs the clusters and works with you to get your workload
running on them.

---

## The four options, side by side

| | own server | virtual machine | container | Kubernetes |
| --- | --- | --- | --- | --- |
| survives a dead server | no | yes, after a restart | no | yes, with no outage |
| you maintain an operating system | yes | yes | yes, the VM's | no |
| reproducible from a recipe | no | partly (snapshots) | yes | yes |
| a crash fixes itself | no | no | no | yes |
| running more copies | buy hardware | build more VMs | set up a load balancer | change one number |
| new concepts to learn | none | a few | some | the most |

Each step trades a little more to learn for a lot less to maintain. The rest of
this workshop is about making that learning as cheap as possible.

---

## The cloud, and "cloud native"

**"The cloud" is a collection of servers that someone else runs for you**, with
software on top that lets you use them. AWS, Azure, Google Cloud and Oracle are
public clouds. CIRRUS is an **on-premises** cloud: the same idea, but the servers
are in NCAR's own data centres and run by NCAR staff.

**Cloud native** is a way of building workloads so they can run on *any* cloud.
The cloud is *where* a workload runs; cloud native is *how it is built*.
Containers plus Kubernetes are what make that possible: an image and its YAML
run on CIRRUS, on a commercial cloud, or on a university's cluster with few or
no changes. Someone at another institution can take your package and run it
themselves, which makes your work reproducible in a way that a hand-built server
never was.

---

## Your code belongs in git

Once everything — the Dockerfile, the YAML, the application — is code, the code
becomes the most valuable part of the workload. If it lives only on one laptop,
we are back to Option 1: one copy, and when that machine dies or its owner
leaves, it is gone. And a folder full of `analysis_v2_FINAL_fixed_REALLY_FINAL.py`
makes it hard to know which version is the real one.

**Git** is a version control system: it tracks every change to a set of files
over time. Words you will need:

* A **commit** is a saved snapshot of your files, with a short message saying what
  changed and who changed it. It is the same idea as a VM snapshot, but for code:
  if something breaks, you can see exactly what changed and go back to a version
  that worked.
* A **repository** (or *repo*) is a project under git. Uploading it to a service
  like GitHub, GitLab or Gitea (a **push**) gives everyone one shared copy;
  downloading it (a **clone**) gives you your own.
* A **branch** is a separate line of work. You make changes there without
  affecting the main version that everyone else depends on.
* A **pull request** (*PR*) asks to merge a branch into the main version.
  Collaborators review the changes, comment on them and approve them first. It is
  like tracked changes in a shared document, except every edit is kept
  permanently and many people can work on different changes at once.

You do not have to love the command line to use git. VS Code, for one, has
built-in source control that covers the everyday commands with buttons.

---

## GitOps: the repository runs the show

Put those together and you get **GitOps**: the git repository holds the complete
description of your workload, and a tool keeps the running system matching it.

This is a form of **CI/CD** — *continuous integration and continuous delivery*.
That just means changes are saved to git often, and tested and deployed often,
by automation rather than by hand.

On CIRRUS, the tool is **Argo CD**. It watches your repository and makes the
cluster match what is in it:

```
  you ──▶ pull request ──▶ review ──▶ merge
                                        │
                                        ▼
                                    git repo  ◀──── Argo CD watches
                                                         │
                                                         ▼
                                              CIRRUS runs what the repo says
```

The repository becomes the **single source of truth**. A merged change reaches a
development or production copy of your workload within minutes. Every change is
reviewed and recorded, so "why is it like this?" is answered by the history. And
other researchers can deploy the same workload straight from your repository.

This is how GDEX itself is run. It was moved off legacy virtual machines onto
CIRRUS by developing and testing it there while the old production instance kept
serving; switching over was just pointing the URLs at the new copy. Developers
now each get their own development copy, and updating production is a pull
request.

---

## Where CIRRUS fits, and what you do not have to learn yet

**CIRRUS is NCAR's Kubernetes platform:** clusters at two NCAR data centres,
plus the services an always-on workload needs, run by the CIRRUS team.
[Page 1](01-orientation.ipynb) covers the details.

The split of work is the important part:

| the CIRRUS team | you |
| --- | --- |
| runs and upgrades the clusters | your application's code |
| runs the shared services: registry, Argo CD, secrets, monitoring | how much CPU and memory it asks for |
| handles certificates, DNS names, and public or internal access | keeping it working when it is restarted or moved |
| helps you package and deploy your workload | |

**You do not need to learn Kubernetes up front to get started.** Getting a
workload onto CIRRUS is a conversation with the team. There is currently no
cost and no allocation request process, and the team's time does not come out of
your project's budget. A typical path:

1. **Talk to the team** about what you want to run.
2. **Get a namespace.** If you have used HPC, think of it as an allocation: your
   own area of the cluster, with access for you and your team and limits on how
   much you can use.
3. **Put the code in a git repository**, if it is not in one already.
4. **Package it in a container** — [page 2](02-containers.ipynb).
5. **Describe how it runs**, in Kubernetes YAML — [page 3](03-kubernetes.ipynb) —
   bundled as a Helm chart — [page 4](04-helm.ipynb).
6. **Connect the repository to Argo CD**, so merges deploy themselves —
   [page 5](05-argocd.ipynb).
7. **Automate tests and image builds**, if the code needs them —
   [page 8](08-github-actions.ipynb).

The team will help with every one of those steps. This workshop exists so you
can do as much of it yourself as you want to, and go as deep as you like over
time.

---

## Words you will meet

| word | in one line | covered on |
| --- | --- | --- |
| server | a computer that answers requests from other computers | this page |
| virtual machine (VM) | a software computer sharing physical hardware with others | this page |
| container | your code packaged with everything it needs to run | [page 2](02-containers.ipynb) |
| image | the built, unchangeable package a container runs from | [page 2](02-containers.ipynb) |
| registry | an online library of images; CIRRUS runs one called Harbor | [page 2](02-containers.ipynb) |
| Kubernetes (K8s) | software that runs containers across many servers and keeps them running | [page 3](03-kubernetes.ipynb) |
| cluster | a group of servers run as one | [page 1](01-orientation.ipynb) |
| namespace | your own area of the cluster; on CIRRUS, like an allocation | [page 1](01-orientation.ipynb) |
| YAML | the plain-text format Kubernetes descriptions are written in | [page 3](03-kubernetes.ipynb) |
| Helm chart | a bundle of YAML files for one application, with settings you can change | [page 4](04-helm.ipynb) |
| git, commit, repo, branch, pull request | version control, and the workflow built on it | this page |
| GitOps, Argo CD | the repository describes the workload, and Argo CD makes the cluster match | [page 5](05-argocd.ipynb) |
| CI/CD | automation that tests and deploys changes as they are made | [page 8](08-github-actions.ipynb) |

---

## Check yourself

1. What is the difference between a batch job and an always-on workload? Which
   one is CIRRUS for?
2. A VM survives its server dying. What still goes wrong for the workload when it
   does?
3. Containers fixed the "OS updates break my workload" problem. What problem did
   they leave unsolved, and what solves it?
4. What does Kubernetes need from you, and what does it do with it?
5. Of the three costs of Kubernetes, which one does the CIRRUS team take on?
6. Why does a workload's code belong in a git repository rather than on the
   server that runs it?

---

← [Start here](README.md) · next: [1. Orientation](01-orientation.ipynb)